# 25 · Explicit Physical-State Gray-Box Layer — H1

This notebook implements the pre-declared Phase-H physical layer. It **does not change the frozen v3 operational policy**. It verifies the 16-feature SHARP contract, builds interpretable magnetic-state axes and temporal evolution, fits a low-capacity physical logistic reference using early training support only, fits a training-only physical-state applicability model, and exports diagnostics for later H2 policy development.

**Evidence boundary:** Cycle-25 and 2026 outcomes have already been seen in A–G. Any later-regime result produced here is post-hoc and cannot be presented as untouched confirmation.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os
import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

ROOT = Path("/home/abmoses2000")
REPO = Path.cwd()
if not (REPO/"results"/"dataset_package_20261002"/"manifest.json").exists():
    # Notebook is normally launched from the repository root.
    for p in [Path("/home/abmoses2000/gray-box-solar-flare-forecasting"), Path("/home/abmoses2000/gray_box_solar_flare_forecasting")]:
        if (p/"results"/"dataset_package_20261002"/"manifest.json").exists():
            REPO=p; break

MANIFEST = REPO/"results"/"dataset_package_20261002"/"manifest.json"
MASTER = ROOT/"graybox_aia72_master_predictions_v2_20261005"/"graybox_aia72_master_predictions.csv.gz"
SHARP_PRED = ROOT/"sharp72_predictions_frozen_20261002.csv.gz"
OUT = ROOT/"graybox_aia72_physical_state_h1_v1_20261006"
assert MANIFEST.is_file(), f"Missing dataset manifest: {MANIFEST}"
assert MASTER.is_file(), f"Missing frozen master predictions: {MASTER}"
assert SHARP_PRED.is_file(), f"Missing frozen SHARP predictions: {SHARP_PRED}"
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

DATA_CANDIDATES=[
    Path(os.environ.get("GRAYBOX_DATASET_ROOT","")) if os.environ.get("GRAYBOX_DATASET_ROOT") else None,
    REPO/"data"/"processed"/"training_snapshot_20261002"/"gray_box_aligned_v1",
    REPO/"data"/"processed"/"gray_box_aligned_v1",
    ROOT/"gray-box-solar-flare-forecasting"/"data"/"processed"/"training_snapshot_20261002"/"gray_box_aligned_v1",
    ROOT/"gray-box-solar-flare-forecasting"/"data"/"processed"/"gray_box_aligned_v1",
]
DATA_CANDIDATES=[p for p in DATA_CANDIDATES if p is not None]
DATA=next((p for p in DATA_CANDIDATES if (p/"sharp.npy").is_file() and (p/"input_index.csv.gz").is_file()),None)
if DATA is None:
    raise FileNotFoundError(
        "Aligned SHARP package not found. Set GRAYBOX_DATASET_ROOT to the directory containing "
        "sharp.npy and input_index.csv.gz. Do not substitute a different tensor."
    )

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b""): h.update(chunk)
    return h.hexdigest()

manifest=json.loads(MANIFEST.read_text())
EXPECTED_FEATURES=["MEANGBZ","MEANGAM","MEANGBT","MEANGBH","MEANJZD","TOTUSJZ","MEANALP","MEANJZH","ABSNJZH","SAVNCPP","MEANSHR","SHRGT45","R_VALUE","USFLUX","TOTPOT","TOTUSJH"]
assert manifest["feature_order"]==EXPECTED_FEATURES, "SHARP feature order changed"
assert manifest["sharp_shape"]==[113433,3,16], "SHARP shape contract changed"
expected_sha=manifest["outputs"]["sharp.npy"]["sha256"]
assert sha256(DATA/"sharp.npy")==expected_sha, "sharp.npy hash mismatch"

MASTER_SHA="17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"
SHARP_PRED_SHA="7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"
assert sha256(MASTER)==MASTER_SHA, "Master prediction hash changed"
assert sha256(SHARP_PRED)==SHARP_PRED_SHA, "Frozen SHARP prediction hash changed"
print("Verified frozen inputs.")
print("Dataset:",DATA)
print("Output:",OUT)


In [ ]:
X=np.load(DATA/"sharp.npy",mmap_mode="r")
idx=pd.read_csv(DATA/"input_index.csv.gz",low_memory=False)
master=pd.read_csv(MASTER,low_memory=False)
sharp_pred=pd.read_csv(SHARP_PRED,low_memory=False)

assert len(X)==len(idx)==113433
assert "forecast_case_id" in master.columns and "forecast_case_id" in sharp_pred.columns
master["forecast_case_id"]=master["forecast_case_id"].astype(str)
sharp_pred["forecast_case_id"]=sharp_pred["forecast_case_id"].astype(str)

# Identify the dataset-to-master join key without guessing silently.
join_candidates=[c for c in ["forecast_case_id","source_sample_id","case_id"] if c in idx.columns and c in master.columns]
if not join_candidates:
    raise ValueError(f"No shared identity key between input_index and master. index={idx.columns.tolist()} master={master.columns.tolist()}")
join_key=join_candidates[0]
idx[join_key]=idx[join_key].astype(str)
master[join_key]=master[join_key].astype(str)

# tensor_row is authoritative when present; otherwise input_index row order is the tensor row.
if "tensor_row" in idx.columns:
    tensor_row=pd.to_numeric(idx["tensor_row"],errors="raise").astype(int).to_numpy()
    assert np.array_equal(np.sort(tensor_row),np.arange(len(X))), "tensor_row is not a complete permutation"
else:
    idx=idx.copy(); idx["tensor_row"]=np.arange(len(idx),dtype=int)

# Physical families are frozen before outcomes are inspected.
FAMILIES={
 "flux_pil":["USFLUX","R_VALUE"],
 "energy_shear":["TOTPOT","MEANSHR","SHRGT45"],
 "current_helicity":["TOTUSJZ","TOTUSJH","ABSNJZH","SAVNCPP","MEANJZD","MEANJZH","MEANALP"],
 "gradients":["MEANGBZ","MEANGBH","MEANGBT"],
 "inclination":["MEANGAM"],
}
SIGNED_MAG={"MEANJZD","MEANJZH","MEANALP"}
fidx={f:i for i,f in enumerate(EXPECTED_FEATURES)}

# Recover issue time and an early training-reference definition from the dataset index.
time_col=next((c for c in ["issue_utc","T_REC_dt","T_REC"] if c in idx.columns),None)
if time_col is None:
    raise ValueError("input_index does not contain a recoverable issue time")
issue=pd.to_datetime(idx[time_col],utc=True,errors="coerce")
assert issue.notna().all(), f"Bad issue times: {issue.isna().sum()}"
train_ref=(issue.dt.year<=2013).to_numpy()
assert train_ref.sum()>1000, f"Training reference unexpectedly small: {train_ref.sum()}"

# Transform each raw feature independently using training-only robust statistics.
# Magnitude-like quantities are log1p(abs(.)); MEANGAM is kept signed/non-log because no monotonic risk direction is imposed.
Z=np.empty_like(np.asarray(X),dtype=np.float64)
transform_rows=[]
for j,f in enumerate(EXPECTED_FEATURES):
    raw=np.asarray(X[:,:,j],dtype=float)
    if f=="MEANGAM":
        t=raw.copy(); transform="identity"
    else:
        t=np.log1p(np.abs(raw)); transform="log1p_abs"
    ref=t[train_ref,:].reshape(-1)
    med=float(np.nanmedian(ref))
    q1,q3=np.nanpercentile(ref,[25,75]); iqr=float(q3-q1)
    if not np.isfinite(iqr) or iqr<=1e-12:
        raise ValueError(f"Degenerate training IQR for {f}: {iqr}")
    Z[:,:,j]=np.clip((t-med)/iqr,-10,10)
    transform_rows.append({"feature":f,"transform":transform,"train_median":med,"train_iqr":iqr})

pd.DataFrame(transform_rows).to_csv(OUT/"physical_feature_transform.csv",index=False)

# Family state per history slot = median robust z across the predeclared members.
family_slot={}
for fam,features in FAMILIES.items():
    jj=[fidx[f] for f in features]
    family_slot[fam]=np.nanmedian(Z[:,:,jj],axis=2)

# Build MPSV: latest level, net change and linear slope for every family.
# Slot coordinates are fixed at [-2,-1,0] in 96-minute steps.
t=np.array([-2.,-1.,0.])
tc=t-t.mean(); denom=float((tc**2).sum())
rows={}
for fam,A in family_slot.items():
    latest=A[:,2]
    net=A[:,2]-A[:,0]
    slope=((A-A.mean(axis=1,keepdims=True))*tc[None,:]).sum(axis=1)/denom
    rows[f"{fam}__latest"]=latest
    rows[f"{fam}__net_change"]=net
    rows[f"{fam}__slope_per96m"]=slope

phys=pd.DataFrame(rows)
MPSV_COLS=phys.columns.tolist()
mpsv=phys.to_numpy(float)
assert np.isfinite(mpsv).all(), "Nonfinite MPSV"

# Fit training-only physical applicability in the transparent low-dimensional state.
lw=LedoitWolf().fit(mpsv[train_ref])
delta=mpsv-lw.location_
md2=np.einsum("ij,jk,ik->i",delta,lw.precision_,delta)
ood_q99=float(np.quantile(md2[train_ref],0.99))
phys["physical_md2"]=md2
phys["physical_ood_q99"]=md2>ood_q99

# Low-capacity interpretable physical probability uses training labels only.
# Labels are joined from frozen SHARP predictions to avoid reconstructing target semantics.
sp=sharp_pred[["forecast_case_id","label","probability_gru_mean"]].copy()
sp["forecast_case_id"]=sp["forecast_case_id"].astype(str)
base=idx[[join_key,"tensor_row"]].merge(master[[join_key,"forecast_case_id"]],on=join_key,how="inner",validate="one_to_one")
base=base.merge(sp,on="forecast_case_id",how="left",validate="one_to_one")
if base["label"].isna().any():
    # training cases are absent from the 71,010-case multimodal master; get labels from dataset arrays when possible.
    yarr=np.load(DATA/"y_primary_72h.npy")
    known=np.load(DATA/"known_primary_72h.npy").astype(bool)
    base["label_dataset"]=yarr[base["tensor_row"].to_numpy(int)]
    base["known_dataset"]=known[base["tensor_row"].to_numpy(int)]
else:
    base["label_dataset"]=base["label"]
    base["known_dataset"]=True

# Build a full tensor-row table first, then attach nontraining frozen predictions.
full=idx[[join_key,"tensor_row"]].copy()
full["issue_dt"]=issue
full["year"]=issue.dt.year.to_numpy()
for c in phys.columns:
    full[c]=phys[c].to_numpy()
yarr=np.load(DATA/"y_primary_72h.npy")
known=np.load(DATA/"known_primary_72h.npy").astype(bool)
full["label"]=yarr
full["known_label"]=known
fit_mask=train_ref & known & np.isin(yarr,[0,1])
assert fit_mask.sum()>1000 and (yarr[fit_mask]==1).sum()>10, "Insufficient training support"

logit=LogisticRegression(C=1.0,solver="lbfgs",max_iter=5000,class_weight=None)
logit.fit(mpsv[fit_mask],yarr[fit_mask])
full["physical_probability"]=logit.predict_proba(mpsv)[:,1]

coef=pd.DataFrame({"physical_axis":MPSV_COLS,"coefficient":logit.coef_[0]})
coef.loc[len(coef)]={"physical_axis":"INTERCEPT","coefficient":float(logit.intercept_[0])}
coef.to_csv(OUT/"physical_logistic_coefficients.csv",index=False)

# Merge the 71,010 frozen multimodal rows for consistency diagnostics.
nontrain=master.merge(sharp_pred,on="forecast_case_id",how="inner",suffixes=("_aia","_sharp"),validate="one_to_one")
label_a="label_aia" if "label_aia" in nontrain.columns else "label"
role_col="role_aia" if "role_aia" in nontrain.columns else "role"
nontrain["label_final"]=pd.to_numeric(nontrain[label_a],errors="raise").astype(int)
nontrain["role_final"]=nontrain[role_col].astype(str)
nontrain["aia_probability"]=pd.to_numeric(nontrain["probability"],errors="coerce")
nontrain["sharp_probability"]=pd.to_numeric(nontrain["probability_gru_mean"],errors="coerce")
nontrain["fusion_probability"]=0.5*nontrain["aia_probability"]+0.5*nontrain["sharp_probability"]

nt=full.merge(nontrain,on=join_key,how="inner",suffixes=("","_pred"),validate="one_to_one")
eps=1e-6
def logodds(p):
    q=np.clip(np.asarray(p,float),eps,1-eps)
    return np.log(q/(1-q))
nt["physical_fusion_logit_gap"]=np.abs(logodds(nt["physical_probability"])-logodds(nt["fusion_probability"]))

# Regime labels are descriptive evidence labels, not new independent splits.
def regime(r):
    if r["role_final"]=="retrospective_cycle25": return "cycle25_2021_2025_posthoc"
    if r["role_final"]=="supplementary_2026": return "supplementary_2026_posthoc"
    if int(r["year"])<=2019: return "cycle24_earlier_development"
    return "other_or_gap"
nt["physical_regime"]=nt.apply(regime,axis=1)

# Frozen H1 thresholds that use no later outcomes.
consistency_ref=nt[nt["physical_regime"]=="cycle24_earlier_development"]["physical_fusion_logit_gap"].dropna().to_numpy()
consistency_q90=float(np.quantile(consistency_ref,0.90))
nt["physical_inconsistency_q90"]=nt["physical_fusion_logit_gap"]>consistency_q90

# Export no policy change; these are diagnostics for H2.
keep=[join_key,"forecast_case_id","issue_dt","year","role_final","label_final",
      "physical_probability","physical_md2","physical_ood_q99",
      "fusion_probability","sharp_probability","aia_probability",
      "physical_fusion_logit_gap","physical_inconsistency_q90","physical_regime"]+MPSV_COLS
nt[keep].to_csv(OUT/"physical_state_case_table.csv.gz",index=False)

freeze={
 "status":"PHASE_H1_PHYSICAL_STATE_CONSTRUCTED",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "policy_changed":False,
 "feature_order":EXPECTED_FEATURES,
 "physical_families":FAMILIES,
 "mpsv_columns":MPSV_COLS,
 "training_reference":"issue year <= 2013, known primary 72h labels only for physical logistic fit",
 "physical_ood_q99":ood_q99,
 "consistency_q90_cycle24_development":consistency_q90,
 "later_regime_evidence":"posthoc_already_seen_in_A_to_G",
 "source_sha256":{
   "sharp_tensor":sha256(DATA/"sharp.npy"),
   "master_predictions":sha256(MASTER),
   "sharp_predictions":sha256(SHARP_PRED),
 },
}
(OUT/"h1_freeze.json").write_text(json.dumps(freeze,indent=2)+"\n")
print(json.dumps(freeze,indent=2))
print("\nCoefficients:\n",coef.to_string(index=False))
print("\nH1 outputs:",OUT)


In [ ]:
# Descriptive summaries only. No thresholds are changed from these later-regime results.
summary=[]
for reg,g in nt.groupby("physical_regime",sort=False):
    y=g["label_final"].to_numpy(int)
    p=g["physical_probability"].to_numpy(float)
    summary.append({
      "regime":reg,
      "cases":len(g),
      "positives":int(y.sum()),
      "prevalence":float(y.mean()),
      "physical_ap":float(average_precision_score(y,p)) if len(np.unique(y))==2 else np.nan,
      "physical_auc":float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
      "physical_brier":float(brier_score_loss(y,p)),
      "physical_ood_rate_q99":float(g["physical_ood_q99"].mean()),
      "median_physical_md2":float(g["physical_md2"].median()),
      "median_physical_fusion_logit_gap":float(g["physical_fusion_logit_gap"].median()),
      "physical_inconsistency_rate_q90":float(g["physical_inconsistency_q90"].mean()),
    })
summary=pd.DataFrame(summary)
summary.to_csv(OUT/"physical_regime_summary.csv",index=False)
print(summary.to_string(index=False))

# Risk association: can physical distance/consistency rank frozen fusion errors?
# This is descriptive; H2 decides whether a signal is safe enough to promote.
risk=[]
for reg,g in nt.groupby("physical_regime",sort=False):
    z=g[np.isfinite(g["fusion_probability"])].copy()
    if len(z)==0: continue
    # use frozen alarm threshold only if available later in H2; here use 0.5 solely for probability-error ranking
    sqerr=(z["fusion_probability"].to_numpy(float)-z["label_final"].to_numpy(int))**2
    for metric in ["physical_md2","physical_fusion_logit_gap"]:
        x=z[metric].to_numpy(float)
        if np.nanstd(x)==0: corr=np.nan
        else: corr=float(pd.Series(x).corr(pd.Series(sqerr),method="spearman"))
        risk.append({"regime":reg,"metric":metric,"spearman_with_fusion_squared_error":corr})
risk=pd.DataFrame(risk)
risk.to_csv(OUT/"physical_risk_association.csv",index=False)
print("\nRisk association:\n",risk.to_string(index=False))

print("\nIMPORTANT: H1 does not modify NORMAL/DEGRADED/ABSTAIN routing. Review these outputs before H2.")


## Stop point

Do **not** add the physical signal to routing from this notebook. First inspect the H1 coefficient signs, physical-axis distributions, later-regime OOD rates, consistency-gap behaviour and positive-class support. H2 must then predeclare any candidate physical gate using earlier development/policy support only. A null or harmful physical signal is a valid outcome and should remain diagnostic rather than being forced into the operational state machine.
